In [ ]:
import numpy as numpy
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f"{x:.2f}")
sns.set_theme(style="darkgrid")

plt.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

RANDOM_STATE = 42
CSV_PATH = "Mall_Customers.csv"

In [ ]:
df = pd.read_csv(CSV_PATH)

In [ ]:
print("Shape:", df.shape)

In [ ]:
df.head()

In [ ]:
customer_id = df["CustomerID"]

In [ ]:
df = df.drop(columns=["CustomerID"])

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.columns:
    print(df[col].value_counts().head(10))

In [ ]:
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()
print("Number of duplicate rows:", num_duplicates)

In [ ]:
plt.figure(figsize=(5, 3))
sns.countplot(x="Gender", data=df)
plt.title("Gender Distribution")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6, 4))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(col, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.scatterplot(
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    data=df,
    s=70
)
plt.title("Income vs Spending Score (Raw Data)")
plt.show()

In [ ]:
df.columns

In [ ]:
columns_to_select = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[columns_to_select]

In [ ]:
X.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled[:5]

In [ ]:
wcss = []
K_range = range(1, 11)

for K in K_range:
    kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(K_range), wcss, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method for Optimal K")
plt.show()

In [ ]:
wcss

In [ ]:
K_final = 5

In [ ]:
Kmeans_final = KMeans(n_clusters=K_final, random_state=RANDOM_STATE, n_init=10)
Kmeans_final.fit(X_scaled)

In [ ]:
clusters = Kmeans_final.predict(X_scaled)
clusters

In [ ]:
df_clusters = df.copy(deep=True)
df_clusters

In [ ]:
df_clusters["Cluster"] = clusters
df_clusters[customer_id] = customer_id
df_clusters.head()

In [ ]:
K_means_score = silhouette_score(X_scaled, df_clusters["Cluster"])
print("K-Means Silhouette Score:", round(K_means_score, 3))

In [ ]:
sil_scores = []

for K in range(2,11):
    model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init =10)
    cluster_labels = model.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, cluster_labels)
    sil_scores.append(sil)

plt.figure(figsize=(6, 4))
plt.plot(range(2, 11), sil_scores, marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores vs K")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    x=df_clusters["Annual Income (k$)"],
    y=df_clusters["Spending Score (1-100)"],
    hue=df_clusters["Cluster"],
    palette="tab10",
)

plt.title("K-Means Clusters (Annual Income vs Spending Score)")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Cluster")
plt.show()

In [ ]:
df.columns

In [ ]:
profile_cols = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]

clusters_sizes = df_clusters["Cluster"].value_counts().sort_index()
clusters_profile_mean = df_clusters.groupby("Cluster")[profile_cols].mean()
clusters_profile_median = df_clusters.groupby("Cluster")[profile_cols].median()

print("Clusters Sizes:")
print(clusters_sizes)

print("\nCluster Profile (Mean):")
print(clusters_profile_mean)

print("\nCluster Profile (Median):")
print(clusters_profile_median)

In [ ]:
df_clusters

In [ ]:
def assign_customer_segment(income_K, spending_score, scaler, model):
    new_point = pd.DataFrame(
        [[income_K, spending_score]],
        columns=["Annual Income (k$)", "Spending Score (1-100)"]
    )

    new_point_scaled = scaler.transform(new_point)
    cluster_id = model.predict(new_point_scaled)[0]
    return cluster_id

In [ ]:
new_cluster = assign_customer_segment(
    income_K=60,
    spending_score=65,
    scaler=scaler,
    model=Kmeans_final
)

print("The new customer belongs to Cluster:", new_cluster)